In [56]:
import tkinter as tk
from tkinter import ttk,messagebox
import math

try:
    import numpy as np
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except:
    np = None
    plt = None
    MATPLOTLIB_AVAILABLE = False


class UltimateCalculator:

    def __init__(self, root):
        self.root = root
        self.root.title("Ultimate Multi Calculator")
        self.root.geometry("400x500")
        self.root.resizable(False, False)

        self.expression = ""
        self.memory = 0
        self.history = []
        self.theme ="dark"

        # ===== DISPLAY =====
        self.display = tk.Entry(
            root,
            font=("Consolas",20, "bold"),
            justify="right",
            bd=8,
            relief="sunken",
            bg="black",
            fg="#00FF66",
            insertbackground="white"
        )
        self.display.pack(fill="x", padx=10, pady=5)

        # ===== THEME BUTTON =====
        self.theme_btn = tk.Button(
            root,
             text="🌙 Dark Mode",
             #text="☀ Light Mode",
             font=("Arial", 11, "bold"),
            command=self.toggle_theme
        )
        self.theme_btn.pack(pady=2)

        # ===== NOTEBOOK =====
        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True)

        self.basic_tab = tk.Frame(self.notebook)
        self.fin_tab = tk.Frame(self.notebook)
        self.graph_tab = tk.Frame(self.notebook)
        self.hist_tab = tk.Frame(self.notebook)

        self.notebook.add(self.basic_tab, text="Scientific")
        self.notebook.add(self.fin_tab, text="Financial")
        self.notebook.add(self.graph_tab, text="Graph")
        self.notebook.add(self.hist_tab, text="History")

        self.create_scientific()
        self.create_financial()
        self.create_graph()
        self.create_history()

        self.apply_theme()

          

    # ================= THEME =================
    def toggle_theme(self):
        self.theme = "light" if self.theme == "dark" else "dark"
        self.theme_btn.config(
            text="☀ Light Mode" if self.theme == "light" else "🌙 Dark Mode"
        )
        self.apply_theme()

    def apply_theme(self):
        if self.theme =="dark":     
          bg ="#000000"
          display_bg ="#171616"
          display_fg ="#00FF66"
          tab_bg ="#0B0D0F"
        else:
          bg = "#919495"
          display_bg = "#FFFFFF"
          display_fg = "#000000"
          tab_bg = "#D5DBDB"

        self.root.config(bg=bg)
        self.display.config(bg=display_bg, fg=display_fg)

        for t in [self.basic_tab, self.fin_tab, self.graph_tab, self.hist_tab]:
            t.config(bg=tab_bg)

    # ================= INPUT =================
    def press(self, v):
        self.expression += str(v)
        self.update_display(self.expression)

    def update_display(self, text):
        self.display.delete(0, tk.END)
        self.display.insert(tk.END, text)

    def clear(self):
        self.expression = ""
        self.update_display("")

    def back(self):
        self.expression = self.expression[:-1]
        self.update_display(self.expression)
#------------------------------------------------------------------
                #=============================================================
    # ================= CALCULATE =================
    
    def calculate(self):
        

        try:
            if not self.expression:
                return

            expr = self.expression
            expr = expr.replace("π", str(math.pi))
            expr = expr.replace("e", str(math.e))
            expr = expr.replace("^", "**")

            import re
            
            pattern = r'(\d+\.?\d*)([\+\-\*/])(\d+\.?\d*)%'

            match = re.search(pattern, expr)

            if match:
                a = float(match.group(1))
                op = match.group(2)
                p = float(match.group(3))
                    

                if op == "+":
                    result = a + (a * p / 100)
                elif op == "-":
                    result = (a - a * p / 100)
                elif op == "*":
                    result = (a * p / 100)
                elif op == "/":
                    result = (a / p / 100)
            else:
                expr = expr.replace("%","/100")
                result = eval(expr)

            self.history.append(f"{self.expression} = {result}")
            self.update_display(result)
            self.expression = str(result)
            self.update_history()

        except:
            self.update_display("Error")
            self.expression = ""

    # ================= SCIENTIFIC =================
    def sci(self, f):
        try:
            v = float(self.display.get())

            ops = {
                "sin": math.sin(math.radians(v)),
                "cos": math.cos(math.radians(v)),
                "tan": math.tan(math.radians(v)),
                "sqrt": math.sqrt(v),
                "log": math.log10(v),
                "ln": math.log(v),
                "x²": v ** 2,
                "x³": v ** 3,
                "1/x": 1 / v if v != 0 else 0,
                "fact": math.factorial(int(v)) if v >= 0 and v == int(v) else 0,
            }

            r = ops[f]
            self.update_display(r)
            self.expression = str(r)

        except:
            messagebox.showerror("Error", "Invalid Input")

    # ================= MEMORY =================
    def Mplus(self): self.memory += float(self.display.get())
    def Mminus(self): self.memory -= float(self.display.get())

    def MR(self):
        self.update_display(self.memory)
        self.expression = str(self.memory)

    def MC(self):
        self.memory = 0

    # ================= SCI UI =================
    def create_scientific(self):
        f = self.basic_tab

        for i in range(8):
            f.grid_rowconfigure(i, weight=1)
        for j in range(5):
            f.grid_columnconfigure(j, weight=1)

        buttons = [
            ["7","8","9","%","sin"],
            ["4","5","6","*","cos"],
            ["1","2","3","-","tan"],
            ["0",".","+","/","sqrt"],
            ["(",")","^","1/x","log"],
            ["π","e","ln","x²","x³"],
            ["AC","fact","=","⌫"]
        ]

        for r, row in enumerate(buttons):
            for c, t in enumerate(row):

                btn = tk.Button(
                    f,
                    text=t,
                    font=("Segoe UI", 14, "bold"),
                    relief="flat",
                    bd=0,
                    cursor="hand2"
                )

                # COLORS
                if t in "0123456789.":
                    btn.config(bg="#B835E8")

                elif t in ["+","-","*","/","^","(",")","1/x","%"]:
                    btn.config(bg="#99D100")

                elif t == "fact":
                    btn.config(bg="#155704")

                elif t in ["sin","cos","tan","sqrt","log","ln","x²","x³","π","e"]:
                    btn.config(bg="#11A3E7")

                elif t == "=":
                    btn.config(bg="#EA0A73", command=self.calculate)

                elif t == "AC":
                    btn.config(bg="#CF200C", command=self.clear)

                elif t == "⌫":
                    btn.config(bg="#CE0808", command=self.back)

                if t not in ["=","AC","⌫"]:
                    if t in ["sin","cos","tan","sqrt","log","ln","x²","x³","1/x","fact"]:
                        btn.config(command=lambda x=t: self.sci(x))
                    else:
                        btn.config(command=lambda x=t: self.press(x))

                btn.grid(row=r, column=c, sticky="nsew", padx=2, pady=2)

    # ================= FINANCIAL =================
    def create_financial(self):
        f = self.fin_tab

        tk.Label(f, text="Principal").pack()
        self.p = tk.Entry(f); self.p.pack()
        
        tk.Label(f, text="Rate").pack()
        self.r = tk.Entry(f); self.r.pack()

        tk.Label(f, text="Years").pack()
        self.t = tk.Entry(f); self.t.pack()

        tk.Button(f, text="SI", command=self.SI).pack()
        tk.Button(f, text="CI", command=self.CI).pack()
        tk.Button(f, text="EMI", command=self.EMI).pack()

        self.fin_res = tk.Label(f, text="Result")
        self.fin_res.pack()

    def SI(self):
        p,r,t = float(self.p.get()), float(self.r.get()), float(self.t.get())
        self.fin_res.config(text=f"SI = {(p*r*t)/100:.2f}")

    def CI(self):
        p,r,t = float(self.p.get()), float(self.r.get()), float(self.t.get())
        self.fin_res.config(text=f"CI = {p*(1+r/100)**t:.2f}")

    def EMI(self):
        p,r,t = float(self.p.get()), float(self.r.get()), float(self.t.get())
        r = r/(12*100); n=t*12
        emi = p/n if r==0 else (p*r*(1+r)**n)/((1+r)**n-1)
        self.fin_res.config(text=f"EMI = {emi:.2f}")

    # ================= GRAPH =================
    def create_graph(self):
        f = self.graph_tab
        tk.Label(f, text="f(x)").pack()
        self.eq = tk.Entry(f)
        self.eq.pack()
        tk.Button(f, text="Plot", command=self.plot).pack()

    def plot(self):
        if not MATPLOTLIB_AVAILABLE:
            messagebox.showerror("Error", "Install numpy & matplotlib")
            return

        try:
            x = np.linspace(-20, 20, 1000)

            allowed = {
                "x": x,
                "np": np,
                "sin": np.sin,
                "cos": np.cos,
                "tan": np.tan,
                "sqrt": np.sqrt,
                "log": np.log,
                "exp": np.exp
            }

            y = eval(self.eq.get(), {"__builtins__": {}}, allowed)

            plt.plot(x, y)
            plt.grid()
            plt.show()

        except:
            messagebox.showerror("Error", "Invalid Equation")

    # ================= HISTORY =================
    def create_history(self):
        self.box = tk.Listbox(self.hist_tab)
        self.box.pack(fill="both", expand=True)

    def update_history(self):
        self.box.delete(0, tk.END)
        for i in self.history:
            self.box.insert(tk.END, i)


# RUN
root = tk.Tk()
app = UltimateCalculator(root)
root.mainloop()

In [25]:
import tkinter as tk
from tkinter import ttk, messagebox
import math

try:
    import numpy as np
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except:
    MATPLOTLIB_AVAILABLE = False


class UltimateCalculator:

    def __init__(self, root):
        self.root = root
        self.root.title("Ultimate Multi Calculator")
        self.root.geometry("356x500")
        self.root.resizable(False, False)
        self.root.configure(bg="#3F4143")

        self.expression = ""
        self.memory = 0
        self.history = []

        # ===== DISPLAY =====
        self.display = tk.Entry(
            root,
            font=("Consolas", 24, "bold"),
            justify="right",
            bg="black",
            fg="#00FF66",
            insertbackground="white",
            bd=8,
            relief="sunken"
        )
        self.display.pack(fill="x", padx=10, pady=10)

        # ===== NOTEBOOK =====
        style = ttk.Style()
        style.theme_use("clam")
        style.configure("TNotebook.Tab", font=("Segoe UI", 10, "bold"), padding=[10, 5])

        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True)

        self.basic_tab = tk.Frame(self.notebook, bg="#ECF0F1")
        self.fin_tab = tk.Frame(self.notebook, bg="#FDF2E9")
        self.graph_tab = tk.Frame(self.notebook, bg="#EBF5FB")
        self.hist_tab = tk.Frame(self.notebook, bg="#F4ECF7")

        self.notebook.add(self.basic_tab, text="Scientific")
        self.notebook.add(self.fin_tab, text="Financial")
        self.notebook.add(self.graph_tab, text="Graph")
        self.notebook.add(self.hist_tab, text="History")

        self.create_scientific()
        self.create_financial()
        self.create_graph()
        self.create_history()

    # ================= INPUT =================
    def press(self, v):
        self.expression += str(v)
        self.update_display(self.expression)

    def update_display(self, text):
        self.display.delete(0, tk.END)
        self.display.insert(tk.END, text)

    def clear(self):
        self.expression = ""
        self.update_display("")

    def back(self):
        self.expression = self.expression[:-1]
        self.update_display(self.expression)

    # ================= CALC =================
    def calculate(self):
        try:
            expr = self.expression

            expr = expr.replace("π", str(math.pi))
            expr = expr.replace("e", str(math.e))
            expr = expr.replace("^", "**")

            # Percentage FIX
    

      # Handle percentage like a normal calculator
    def calculate():
       global expression

    try:
        expr = expression.replace(" ", "")

        # Handle percentage like a normal calculator

        if "+" in expr and expr.endswith("%"):
            num, per = expr.rsplit("+", 1)
            result = float(num) + (float(num) * float(per[:-1]) / 100)

        elif "-" in expr and expr.endswith("%"):
            num, per = expr.rsplit("-", 1)
            result = float(num) - (float(num) * float(per[:-1]) / 100)

        elif "*" in expr and expr.endswith("%"):
            num, per = expr.rsplit("*", 1)
            result = float(num) * (float(per[:-1]) / 100)

        elif "/" in expr and expr.endswith("%"):
            num, per = expr.rsplit("/", 1)
            result = float(num) / (float(per[:-1]) / 100)

        else:
            # Convert standalone percentages
            expr = expr.replace("%", "/100")

            result = eval(
                expr,
                {"__builtins__": None},
                {
                    "pi": math.pi,
                    "e": math.e
                }
            )

        display_var.set(result)
        expression = str(result)

    except Exception:
        display_var.set("Error")
        expression = ""

        #if "+" in expr and expr.endswith("%"):
            #num, per = expr.rsplit("+", 1)
            #result = float(num) + (float(num) * float(per[:-1]) / 100)

        #elif "-" in expr and expr.endswith("%"):
           # num, per = expr.rsplit("-", 1)
            #result = float(num) - (float(num) * float(per[:-1]) / 100)

        #elif "*" in expr and expr.endswith("%"):
            #num, per = expr.rsplit("*", 1)
           # result = float(num) * (float(per[:-1]) / 100)

        #elif "/" in expr and expr.endswith("%"):
           # num, per = expr.rsplit("/", 1)
           # result = float(num) / (float(per[:-1]) / 100)

        #else:
            # Convert standalone percentages
            #expr = expr.replace("%", "/100")

           # result = eval(
              #  expr,
               # {"__builtins__": None},
                #{
                #    "pi": math.pi,
                #    "e": math.e
                #}
           # )

        #if "%" in expr:
        #expression = ""
                ##if re := __import__("re"):
                    #pattern = r'(\d+\.?\d*)([\+\-\*/])(\d+\.?\d*)%'
                    #m = re.search(pattern, expr)
                    #if m:

                       # a = float(m.group(1))
                       # op = m.group(2)
                        #p = float(m.group(3))

                       # if op == "+":
                            #result = a + (a * p / 100)
                       # elif op == "-":
                           # result = a - (a * p / 100)
                        #elif op == "*":
                            #result = a * (p / 100)
                        #elif op == "/":
                            #result = a / (p / 100)
                    #else:
                       # result = eval(expr.replace("%", "/100"))
            #else:
               # result = eval(expr)

    self.history.append(f"{self.expression} = {result}")
    self.update_display(result)
            self.expression = str(result)
            self.update_history()

        except:
            self.update_display("Error")
            self.expression = ""

    # ================= SCIENTIFIC =================
    def sci(self, f):
        try:
            v = float(self.display.get())

            ops = {
                "sin": math.sin(math.radians(v)),
                "cos": math.cos(math.radians(v)),
                "tan": math.tan(math.radians(v)),
                "sqrt": math.sqrt(v),
                "log": math.log10(v),
                "ln": math.log(v),
                "x²": v ** 2,
                "x³": v ** 3,
                "1/x": 1 / v if v != 0 else "Error",
                "fact": math.factorial(int(v)),
            }

            r = ops[f]
            self.update_display(r)
            self.expression = str(r)

        except:
            messagebox.showerror("Error", "Invalid Input")

    # ================= MEMORY =================
    def Mplus(self):
        self.memory += float(self.display.get())

    def Mminus(self):
        self.memory -= float(self.display.get())

    def MR(self):
        self.update_display(self.memory)
        self.expression = str(self.memory)

    def MC(self):
        self.memory = 0

    # ================= SCI BUTTONS =================
    def create_scientific(self):
        frame = self.basic_tab

        buttons = [
            ["7","8","9","/","sin"],
            ["4","5","6","*","cos"],
            ["1","2","3","-","tan"],
            ["0",".","+","=","sqrt"],
            ["(",")","^","%","log"],
            ["π","e","ln","x²","x³"],
            ["AC","1/x","fact","⌫",]
        ]

        for r, row in enumerate(buttons):
            for c, t in enumerate(row):

                btn = tk.Button(frame, text=t, font=("Arial", 11, "bold"),
                                width=6, height=2, relief="raised", bd=3)

                # ===== COLORS =====
                if t in "0123456789.":
                    btn.config(bg="#0A0A0A", fg="white")

                elif t in ["+","-","*","%","/","^","(",")"]:
                    btn.config(bg="#B87408")

                elif t in ["sin","cos","tan","sqrt","log","ln","x²","x³","1/x","fact","π","e"]:
                    btn.config(bg="#04853C")

                elif t == "=":
                    btn.config(bg="#AE277F", command=self.calculate)

                elif t == "AC":
                    btn.config(bg="#D61B06", command=self.clear)

                elif t == "⌫":
                    btn.config(bg="#732694", command=self.back)

                else:
                    btn.config(command=lambda x=t: self.press(x))

                # Commands
                if t not in ["=","AC","⌫"]:
                    if t in ["sin","cos","tan","sqrt","log","ln","x²","x³","1/x","fact"]:
                        btn.config(command=lambda x=t: self.sci(x))
                    else:
                        btn.config(command=lambda x=t: self.press(x))

                btn.grid(row=r, column=c, padx=2, pady=2)

                # Hover
                btn.bind("<Enter>", lambda e,b=btn: b.config(relief="sunken"))
                btn.bind("<Leave>", lambda e,b=btn: b.config(relief="raised"))

    # ================= FINANCIAL =================
    def create_financial(self):
        f = self.fin_tab

        tk.Label(f, text="Principal").pack()
        self.p = tk.Entry(f); self.p.pack()

        tk.Label(f, text="Rate %").pack()
        self.r = tk.Entry(f); self.r.pack()

        tk.Label(f, text="Years").pack()
        self.t = tk.Entry(f); self.t.pack()

        tk.Button(f, text="SI", command=self.SI).pack(pady=2)
        tk.Button(f, text="CI", command=self.CI).pack(pady=2)
        tk.Button(f, text="EMI", command=self.EMI).pack(pady=2)

        self.fin_res = tk.Label(f, text="Result", font=("Arial", 12, "bold"))
        self.fin_res.pack(pady=10)

    def SI(self):
        p,r,t = float(self.p.get()), float(self.r.get()), float(self.t.get())
        self.fin_res.config(text=f"SI = {(p*r*t)/100:.2f}")

    def CI(self):
        p,r,t = float(self.p.get()), float(self.r.get()), float(self.t.get())
        self.fin_res.config(text=f"CI = {p*(1+r/100)**t:.2f}")

    def EMI(self):
        p,r,t = float(self.p.get()), float(self.r.get()), float(self.t.get())
        r = r/(12*100); n=t*12
        emi = p/n if r==0 else (p*r*(1+r)**n)/((1+r)**n-1)
        self.fin_res.config(text=f"EMI = {emi:.2f}")

    # ================= GRAPH =================
    def create_graph(self):
        f = self.graph_tab

        tk.Label(f, text="Enter f(x)").pack()
        self.eq = tk.Entry(f, width=30)
        self.eq.pack()

        tk.Button(f, text="Plot", command=self.plot).pack()

    def plot(self):
        if not MATPLOTLIB_AVAILABLE:
            messagebox.showerror("Error", "Install numpy & matplotlib")
            return

        try:
            x = np.linspace(-20, 20, 1000)

            allowed = {
                "x": x,
                "np": np,
                "sin": np.sin,
                "cos": np.cos,
                "tan": np.tan,
                "sqrt": np.sqrt,
                "log": np.log,
                "exp": np.exp
            }

            y = eval(self.eq.get(), {"__builtins__": {}}, allowed)

            plt.plot(x, y)
            plt.grid()
            plt.title(self.eq.get())
            plt.show()

        except:
            messagebox.showerror("Error", "Invalid Equation")

    # ================= HISTORY =================
    def create_history(self):
        self.box = tk.Listbox(self.hist_tab)
        self.box.pack(fill="both", expand=True)

    def update_history(self):
        self.box.delete(0, tk.END)
        for i in self.history:
            self.box.insert(tk.END, i)


# RUN
root = tk.Tk()
app = UltimateCalculator(root)
root.mainloop()

IndentationError: unindent does not match any outer indentation level (<string>, line 194)

In [22]:
import tkinter as tk
from tkinter import messagebox
import math

# ---------------- MAIN WINDOW ----------------
root = tk.Tk()
root.title("Ultimate Calculator")
root.geometry("500x600")
root.resizable(False, False)
root.configure(bg="#1e1e1e")

expression = ""
memory = 0

# ---------------- DISPLAY ----------------
display_var = tk.StringVar()

display = tk.Entry(
    root,
    textvariable=display_var,
    font=("Arial", 24, "bold"),
    justify="right",
    bd=8,
    relief="groove",
    bg="white"
)
display.pack(fill="x", padx=10, pady=10)

# ---------------- FUNCTIONS ----------------

def press(value):
    global expression
    expression += str(value)
    display_var.set(expression)

def clear():
    global expression
    expression = ""
    display_var.set("")

def backspace():
    global expression
    expression = expression[:-1]
    display_var.set(expression)

def calculate():
    global expression

    try:
        expr = expression

        # Calculator style percentage
        if "+" in expr and "%" in expr:
            num, per = expr.split("+")
            result = float(num) + (float(num) * float(per.replace("%", "")) / 100)

        elif "-" in expr and "%" in expr:
            num, per = expr.split("-")
            result = float(num) - (float(num) * float(per.replace("%", "")) / 100)

        elif "*" in expr and "%" in expr:
            num, per = expr.split("*")
            result = float(num) * (float(per.replace("%", "")) / 100)

        elif "/" in expr and "%" in expr:
            num, per = expr.split("/")
            result = float(num) / (float(per.replace("%", "")) / 100)

        else:
            result = eval(
                expr,
                {"__builtins__": None},
                {
                    "pi": math.pi,
                    "e": math.e
                }
            )

        display_var.set(result)
        expression = str(result)

    except:
        display_var.set("Error")
        expression = ""

# ---------------- SCIENTIFIC ----------------

def scientific(func):
    global expression

    try:
        value = float(display_var.get())

        if func == "sin":
            result = math.sin(math.radians(value))

        elif func == "cos":
            result = math.cos(math.radians(value))

        elif func == "tan":
            result = math.tan(math.radians(value))

        elif func == "sqrt":
            result = math.sqrt(value)

        elif func == "log":
            result = math.log10(value)

        elif func == "ln":
            result = math.log(value)

        elif func == "square":
            result = value ** 2

        elif func == "cube":
            result = value ** 3

        elif func == "fact":
            if value < 0 or int(value) != value:
                raise ValueError
            result = math.factorial(int(value))

        elif func == "inv":
            result = 1 / value

        display_var.set(result)
        expression = str(result)

    except:
        display_var.set("Error")

# ---------------- MEMORY ----------------

def memory_clear():
    global memory
    memory = 0

def memory_recall():
    display_var.set(str(memory))

def memory_add():
    global memory
    try:
        memory += float(display_var.get())
    except:
        pass

def memory_subtract():
    global memory
    try:
        memory -= float(display_var.get())
    except:
        pass

# ---------------- FINANCIAL ----------------

def simple_interest():
    try:
        p = float(principal_entry.get())
        r = float(rate_entry.get())
        t = float(time_entry.get())

        si = (p * r * t) / 100

        messagebox.showinfo(
            "Simple Interest",
            f"Interest = {si:.2f}\nAmount = {p + si:.2f}"
        )

    except:
        messagebox.showerror("Error", "Invalid Input")

def compound_interest():
    try:
        p = float(principal_entry.get())
        r = float(rate_entry.get())
        t = float(time_entry.get())

        amount = p * ((1 + r / 100) ** t)
        ci = amount - p

        messagebox.showinfo(
            "Compound Interest",
            f"CI = {ci:.2f}\nAmount = {amount:.2f}"
        )

    except:
        messagebox.showerror("Error", "Invalid Input")

def emi():
    try:
        P = float(principal_entry.get())
        annual_rate = float(rate_entry.get())
        n = int(time_entry.get())

        r = annual_rate / 12 / 100

        if r == 0:
            emi_value = P / n
        else:
            emi_value = P * r * ((1 + r) ** n) / (((1 + r) ** n) - 1)

        messagebox.showinfo(
            "EMI Calculator",
            f"Monthly EMI = {emi_value:.2f}"
        )

    except:
        messagebox.showerror("Error", "Invalid Input")

# ---------------- BUTTON STYLE ----------------

btn_style = {
    "font": ("Arial", 12, "bold"),
    "width": 5,
    "height": 2,
    "bg": "#2d89ef",
    "fg": "white",
    "activebackground": "#ff9800"
}

# ---------------- MEMORY FRAME ----------------

memory_frame = tk.Frame(root, bg="#1e1e1e")
memory_frame.pack()

for text, cmd in [
    ("MC", memory_clear),
    ("MR", memory_recall),
    ("M+", memory_add),
    ("M-", memory_subtract)
]:
    tk.Button(
        memory_frame,
        text=text,
        command=cmd,
        bg="#9c27b0",
        fg="white",
        width=8
    ).pack(side="left", padx=3, pady=3)

# ---------------- CALCULATOR BUTTONS ----------------

frame = tk.Frame(root, bg="#1e1e1e")
frame.pack()

buttons = [
    ['7', '8', '9', '/', 'AC'],
    ['4', '5', '6', '*', '←'],
    ['1', '2', '3', '-', '%'],
    ['0', '.', '=', '+', '^']
]

for r, row in enumerate(buttons):
    for c, text in enumerate(row):

        if text == "AC":
            cmd = clear
        elif text == "←":
            cmd = backspace
        elif text == "=":
            cmd = calculate
        elif text == "^":
            cmd = lambda: press("**")
        else:
            cmd = lambda t=text: press(t)

        tk.Button(
            frame,
            text=text,
            command=cmd,
            **btn_style
        ).grid(row=r, column=c, padx=4, pady=4)

# ---------------- SCIENTIFIC ----------------

sci = tk.LabelFrame(
    root,
    text="Scientific Functions",
    fg="white",
    bg="#1e1e1e"
)
sci.pack(fill="x", padx=10, pady=5)

scientific_buttons = [
    ("sin", "sin"),
    ("cos", "cos"),
    ("tan", "tan"),
    ("√", "sqrt"),
    ("log", "log"),
    ("ln", "ln"),
    ("x²", "square"),
    ("x³", "cube"),
    ("n!", "fact"),
    ("1/x", "inv")
]

for i, (txt, func) in enumerate(scientific_buttons):
    tk.Button(
        sci,
        text=txt,
        command=lambda f=func: scientific(f),
        bg="#28a745",
        fg="white",
        width=8
    ).grid(row=i // 5, column=i % 5, padx=3, pady=3)

# ---------------- FINANCIAL ----------------

finance = tk.LabelFrame(
    root,
    text="Financial Calculator",
    fg="white",
    bg="#1e1e1e"
)
finance.pack(fill="x", padx=10, pady=5)

tk.Label(finance, text="Principal", bg="#1e1e1e", fg="white").grid(row=0, column=0)
principal_entry = tk.Entry(finance, width=12)
principal_entry.grid(row=0, column=1)

tk.Label(finance, text="Rate %", bg="#1e1e1e", fg="white").grid(row=0, column=2)
rate_entry = tk.Entry(finance, width=12)
rate_entry.grid(row=0, column=3)

tk.Label(finance, text="Time", bg="#1e1e1e", fg="white").grid(row=1, column=0)
time_entry = tk.Entry(finance, width=12)
time_entry.grid(row=1, column=1)

tk.Button(
    finance,
    text="Simple Interest",
    command=simple_interest,
    bg="#ff9800",
    fg="white"
).grid(row=2, column=0, pady=5)

tk.Button(
    finance,
    text="Compound Interest",
    command=compound_interest,
    bg="#ff9800",
    fg="white"
).grid(row=2, column=1, pady=5)

tk.Button(
    finance,
    text="EMI",
    command=emi,
    bg="#ff9800",
    fg="white"
).grid(row=2, column=2, pady=5)

# ---------------- FOOTER ----------------

tk.Label(
    root,
    text="Basic + Scientific + Financial Calculator",
    bg="#1e1e1e",
    fg="cyan",
    font=("Arial", 10, "bold")
).pack(side="bottom", pady=5)

root.mainloop()

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import math

try:
    import matplotlib.pyplot as plt
    import numpy as np
    MATPLOTLIB_AVAILABLE = True
except:
    MATPLOTLIB_AVAILABLE = False


class MultiCalculator:

    def __init__(self, root):
        self.root = root
        self.root.title("Ultimate Multi Calculator")
        self.root.geometry("500x600")
        self.root.resizable(False, False)

        self.expression = ""

        # Display
        self.display = tk.Entry(
            root,
            font=("Arial", 24, "bold"),
            justify="right",
            bd="white"
        )
        self.display.pack(fill="x", padx=10, pady=10)

        # Notebook Tabs
        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True)

        self.basic_tab = tk.Frame(self.notebook)
        self.financial_tab = tk.Frame(self.notebook)
        self.graph_tab = tk.Frame(self.notebook)
        self.history_tab = tk.Frame(self.notebook)

        self.notebook.add(self.basic_tab, text="Scientific")
        self.notebook.add(self.financial_tab, text="Financial")
        self.notebook.add(self.graph_tab, text="Graph")
        self.notebook.add(self.history_tab, text="History")

        self.history = []

        self.create_scientific_calculator()
        self.create_financial_calculator()
        self.create_graph_tab()
        self.create_history_tab()

    # ================= SCIENTIFIC =================

    def press(self, value):
        self.expression += str(value)
        self.display.delete(0, tk.END)
        self.display.insert(tk.END, self.expression)

    def clear(self):
        self.expression = ""
        self.display.delete(0, tk.END)

    def backspace(self):
        self.expression = self.expression[:-1]
        self.display.delete(0, tk.END)
        self.display.insert(tk.END, self.expression)

    def calculate(self):
        try:
            expr = self.expression

            replacements = {
                "π": str(math.pi),
                "e": str(math.e),
                "^": "**"
            }

            for k, v in replacements.items():
                expr = expr.replace(k, v)

            result = eval(expr)

            self.history.append(f"{self.expression} = {result}")

            self.display.delete(0, tk.END)
            self.display.insert(tk.END, result)

            self.expression = str(result)

            self.update_history()

        except:
            self.display.delete(0, tk.END)
            self.display.insert(tk.END, "Error")
            self.expression = ""

    def scientific_function(self, func):
        try:
            value = float(self.display.get())

            operations = {
                "sin": math.sin(math.radians(value)),
                "cos": math.cos(math.radians(value)),
                "tan": math.tan(math.radians(value)),
                "sqrt": math.sqrt(value),
                "log": math.log10(value),
                "ln": math.log(value),
                "x²": value ** 2,
                "1/x": 1 / value,
                "fact": math.factorial(int(value))
            }

            result = operations[func]

            self.display.delete(0, tk.END)
            self.display.insert(tk.END, result)
            self.expression = str(result)

        except:
            messagebox.showerror("Error", "Invalid Input")

    def create_scientific_calculator(self):

        frame = self.basic_tab

        buttons = [
            ["7", "8", "9", "/", "sin"],
            ["4", "5", "6", "*", "cos"],
            ["1", "2", "3", "-", "tan"],
            ["0", ".", "+", "=", "sqrt"],
            ["(", ")", "^", "AC", "log"],
            ["π", "e", "ln", "x²", "1/x"],
            ["fact", "⌫"]
        ]

        for r, row in enumerate(buttons):
            for c, text in enumerate(row):

                btn = tk.Button(
                    frame,
                    text=text,
                    font=("Arial", 12, "bold"),
                    width=7,
                    height=2
                )

                btn.grid(row=r, column=c, padx=3, pady=3)

                if text == "=":
                    btn.config(command=self.calculate)

                elif text == "AC":
                    btn.config(command=self.clear)

                elif text == "⌫":
                    btn.config(command=self.backspace)

                elif text in ["sin", "cos", "tan",
                              "sqrt", "log", "ln",
                              "x²", "1/x", "fact"]:
                    btn.config(
                        command=lambda t=text:
                        self.scientific_function(t)
                    )

                else:
                    btn.config(
                        command=lambda t=text:
                        self.press(t)
                    )

    # ================= FINANCIAL =================

    def create_financial_calculator(self):

        frame = self.financial_tab

        tk.Label(frame, text="Principal").pack(pady=5)
        self.principal = tk.Entry(frame)
        self.principal.pack()

        tk.Label(frame, text="Rate (%)").pack(pady=5)
        self.rate = tk.Entry(frame)
        self.rate.pack()

        tk.Label(frame, text="Years").pack(pady=5)
        self.time = tk.Entry(frame)
        self.time.pack()

        tk.Button(
            frame,
            text="Simple Interest",
            command=self.simple_interest
        ).pack(pady=5)

        tk.Button(
            frame,
            text="Compound Interest",
            command=self.compound_interest
        ).pack(pady=5)

        tk.Button(
            frame,
            text="EMI Calculator",
            command=self.calculate_emi
        ).pack(pady=5)

        self.fin_result = tk.Label(
            frame,
            text="Result",
            font=("Arial", 12, "bold")
        )
        self.fin_result.pack(pady=10)

    def simple_interest(self):
        try:
            p = float(self.principal.get())
            r = float(self.rate.get())
            t = float(self.time.get())

            si = (p * r * t) / 100

            self.fin_result.config(
                text=f"Simple Interest = {si:.2f}"
            )

        except:
            messagebox.showerror("Error", "Invalid Input")

    def compound_interest(self):
        try:
            p = float(self.principal.get())
            r = float(self.rate.get())
            t = float(self.time.get())

            amount = p * (1 + r / 100) ** t

            self.fin_result.config(
                text=f"Amount = {amount:.2f}"
            )

        except:
            messagebox.showerror("Error", "Invalid Input")

    def calculate_emi(self):
        try:
            p = float(self.principal.get())
            annual_rate = float(self.rate.get())

            years = float(self.time.get())

            r = annual_rate / (12 * 100)
            n = years * 12

            emi = (
                p * r * (1 + r) ** n
            ) / (
                (1 + r) ** n - 1
            )

            self.fin_result.config(
                text=f"EMI = {emi:.2f}"
            )

        except:
            messagebox.showerror("Error", "Invalid Input")

    # ================= GRAPH =================

    def create_graph_tab(self):

        frame = self.graph_tab

        tk.Label(
            frame,
            text="Enter equation using x"
        ).pack(pady=10)

        self.graph_entry = tk.Entry(
            frame,
            width=30
        )
        self.graph_entry.pack()

        tk.Button(
            frame,
            text="Plot Graph",
            command=self.plot_graph
        ).pack(pady=10)

    def plot_graph(self):

        if not MATPLOTLIB_AVAILABLE:
            messagebox.showerror(
                "Error",
                "Install matplotlib and numpy"
            )
            return

        equation = self.graph_entry.get()

        try:
            x = np.linspace(-20, 20, 1000)
            y = eval(equation)

            plt.figure(figsize=(6, 4))
            plt.plot(x, y)
            plt.grid(True)
            plt.title(equation)
            plt.xlabel("x")
            plt.ylabel("y")
            plt.show()

        except:
            messagebox.showerror(
                "Error",
                "Invalid Equation"
            )

    # ================= HISTORY =================

    def create_history_tab(self):

        self.history_box = tk.Listbox(
            self.history_tab,
            width=60,
            height=25
        )

        self.history_box.pack(
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

    def update_history(self):

        self.history_box.delete(0, tk.END)

        for item in self.history:
            self.history_box.insert(tk.END, item)


root = tk.Tk()
app = MultiCalculator(root)
root.mainloop()

In [21]:
from tkinter import *
import math

expr = ""

def press(key):
    global expr
    expr += str(key)
    display.set(expr)

def equal():
    global expr
    try:
        if "%" in expr:

            if "-" in expr:
                num, per = expr.split("-")
                per = per.replace("%", "")
                result = float(num) - (float(num) * float(per) / 100)

            elif "+" in expr:
                num, per = expr.split("+")
                per = per.replace("%", "")
                result = float(num) + (float(num) * float(per) / 100)

            elif "*" in expr:
                num, per = expr.split("*")
                per = per.replace("%", "")
                result = float(num) * (float(per) / 100)

            elif "/" in expr:
                num, per = expr.split("/")
                per = per.replace("%", "")
                result = float(num) / (float(per) / 100)

            else:
                result = float(expr.replace("%", "")) / 100

        else:
            result = eval(expr)

        display.set(str(result))
        expr = str(result)

    except:
        display.set("Error")
        expr = ""

def clear():
    global expr
    expr = ""
    display.set("")

def backspace():
    global expr
    expr = expr[:-1]
    display.set(expr)

def square_root():
    global expr
    try:
        result = str(math.sqrt(float(eval(expr))))
        display.set(result)
        expr = result
    except:
        display.set("Error")
        expr = ""

def power():
    press("**")


root = Tk()
root.title("Advanced Calculator")
root.geometry("450x600")
root.configure(bg="grey")

display = StringVar()

entry = Entry(
    root,
    textvariable=display,
    font=("Arial", 22),
    justify="right",
    bd=10
)
entry.grid(row=0, column=0, columnspan=4, ipadx=8, ipady=15, pady=10)

buttons = [
    ('AC', 1, 0), ('⌫', 1, 1), ('%', 1, 2), ('/', 1, 3),
    ('7', 2, 0), ('8', 2, 1), ('9', 2, 2), ('*', 2, 3),
    ('4', 3, 0), ('5', 3, 1), ('6', 3, 2), ('-', 3, 3),
    ('1', 4, 0), ('2', 4, 1), ('3', 4, 2), ('+', 4, 3),
    ('0', 5, 0), ('.', 5, 1), ('=', 5, 2), ('√', 5, 3),
    ('(', 6, 0), (')', 6, 1), ('^', 6, 2)
]

for (text, row, col) in buttons:

    if text == "AC":
        cmd = clear

    elif text == "=":
        cmd = equal

    elif text == "⌫":
        cmd = backspace

    elif text == "√":
        cmd = square_root

    elif text == "^":
        cmd = power

    else:
        cmd = lambda t=text: press(t)

    Button(
        root,
        text=text,
        width=8,
        height=3,
        font=("Arial", 14, "bold"),
        bg="black",
        fg="white",
        command=cmd
    ).grid(row=row, column=col, padx=2, pady=2)

root.mainloop()

In [20]:
import tkinter as tk
from tkinter import messagebox
import math

# ==========================
# Calculator Class
# ==========================
class AdvancedCalculator:
    def __init__(self, root):
        self.root = root
        self.root.title("Advanced Scientific & Financial Calculator")
        self.root.geometry("500x600")
        self.root.resizable(False, False)
        self.root.configure(bg="#1e1e1e")

        self.expression = ""
        self.memory = 0

        # Display
        self.display_var = tk.StringVar()

        display = tk.Entry(
            root,
            textvariable=self.display_var,
            font=("Arial", 24, "bold"),
            bd=10,
            relief=tk.FLAT,
            justify="right",
            bg="#D7D7D7",
            fg="black"
        )
        display.pack(fill="x", padx=10, pady=10, ipady=15)

        # Button Frame
        btn_frame = tk.Frame(root, bg="#1e1e1e")
        btn_frame.pack(expand=True, fill="both")

        buttons = [
            ["MC", "MR", "M+", "M-", "AC"],
            ["sin", "cos", "tan", "√", "π"],
            ["log", "ln", "x²", "x³", "e"],
            ["7", "8", "9", "/", "%"],
            ["4", "5", "6", "*", "("],
            ["1", "2", "3", "-", ")"],
            ["0", ".", "^", "+", "="],
            ["SI", "CI", "EMI", "⌫", ""]
        ]

        for r, row in enumerate(buttons):
            for c, text in enumerate(row):
                if text == "":
                    continue

                bg = "#3a3a3a"

                if text in ["=", "SI", "CI", "EMI"]:
                    bg = "#B800F0"

                elif text in ["AC", "⌫"]:
                    bg = "#f90800"

                elif text in ["sin", "cos", "tan", "√",
                              "log", "ln", "x²", "x³",
                              "π", "e"]:
                    bg = "#0217ff"

                elif text in ["MC", "MR", "M+", "M-"]:
                    bg = "#00ff00"

                btn = tk.Button(
                    btn_frame,
                    text=text,
                    font=("Arial", 15, "bold"),
                    bg=bg,
                    fg="white",
                    bd=0,
                    command=lambda t=text: self.button_click(t)
                )

                btn.grid(
                    row=r,
                    column=c,
                    sticky="nsew",
                    padx=3,
                    pady=3
                )

        for i in range(8):
            btn_frame.rowconfigure(i, weight=1)

        for i in range(5):
            btn_frame.columnconfigure(i, weight=1)

    # ==========================
    # Button Handling
    # ==========================
    def button_click(self, value):

        if value == "=":
            self.calculate()

        elif value == "AC":
            self.expression = ""
            self.display_var.set("")

        elif value == "⌫":
            self.expression = self.expression[:-1]
            self.display_var.set(self.expression)

        elif value == "π":
            self.expression += str(math.pi)
            self.display_var.set(self.expression)

        elif value == "e":
            self.expression += str(math.e)
            self.display_var.set(self.expression)

        elif value == "√":
            self.expression += "math.sqrt("
            self.display_var.set(self.expression)

        elif value == "sin":
            self.expression += "math.sin(math.radians("
            self.display_var.set(self.expression)

        elif value == "cos":
            self.expression += "math.cos(math.radians("
            self.display_var.set(self.expression)

        elif value == "tan":
            self.expression += "math.tan(math.radians("
            self.display_var.set(self.expression)

        elif value == "log":
            self.expression += "math.log10("
            self.display_var.set(self.expression)

        elif value == "ln":
            self.expression += "math.log("
            self.display_var.set(self.expression)

        elif value == "x²":
            self.expression += "**2"
            self.display_var.set(self.expression)

        elif value == "x³":
            self.expression += "**3"
            self.display_var.set(self.expression)

        elif value == "^":
            self.expression += "**"
            self.display_var.set(self.expression)

        # Memory Functions
        elif value == "MC":
            self.memory = 0

        elif value == "MR":
            self.expression += str(self.memory)
            self.display_var.set(self.expression)

        elif value == "M+":
            try:
                self.memory += float(eval(self.expression))
            except:
                pass

        elif value == "M-":
            try:
                self.memory -= float(eval(self.expression))
            except:
                pass

        # Financial Functions
        elif value == "SI":
            self.simple_interest()

        elif value == "CI":
            self.compound_interest()

        elif value == "EMI":
            self.emi_calculator()

        else:
            self.expression += str(value)
            self.display_var.set(self.expression)

    # ==========================
    # Percentage Handling
    # ==========================
    def calculate(self):

        try:
            expr = self.expression

            if "%" in expr:
                expr = self.handle_percentage(expr)

            result = eval(expr)
            self.display_var.set(result)
            self.expression = str(result)

        except Exception:
            self.display_var.set("Error")
            self.expression = ""

    def handle_percentage(self, expr):

        expr = expr.replace("%", "/100")

        return expr

    # ==========================
    # Financial Functions
    # ==========================
    def simple_interest(self):

        try:
            p = float(
                tk.simpledialog.askstring(
                    "Simple Interest",
                    "Principal Amount:"
                )
            )

            r = float(
                tk.simpledialog.askstring(
                    "Simple Interest",
                    "Rate (%):"
                )
            )

            t = float(
                tk.simpledialog.askstring(
                    "Simple Interest",
                    "Time (Years):"
                )
            )

            si = (p * r * t) / 100

            messagebox.showinfo(
                "Simple Interest",
                f"Simple Interest = {si:.2f}"
            )

        except:
            pass

    def compound_interest(self):

        try:
            p = float(
                tk.simpledialog.askstring(
                    "Compound Interest",
                    "Principal Amount:"
                )
            )

            r = float(
                tk.simpledialog.askstring(
                    "Compound Interest",
                    "Rate (%):"
                )
            )

            t = float(
                tk.simpledialog.askstring(
                    "Compound Interest",
                    "Years:"
                )
            )

            amount = p * ((1 + r / 100) ** t)

            ci = amount - p

            messagebox.showinfo(
                "Compound Interest",
                f"Compound Interest = {ci:.2f}"
            )

        except:
            pass

    def emi_calculator(self):

        try:
            loan = float(
                tk.simpledialog.askstring(
                    "EMI",
                    "Loan Amount:"
                )
            )

            rate = float(
                tk.simpledialog.askstring(
                    "EMI",
                    "Annual Interest (%):"
                )
            )

            months = int(
                tk.simpledialog.askstring(
                    "EMI",
                    "Months:"
                )
            )

            monthly_rate = rate / 12 / 100

            emi = (
                loan *
                monthly_rate *
                ((1 + monthly_rate) ** months)
            ) / (
                ((1 + monthly_rate) ** months) - 1
            )

            messagebox.showinfo(
                "EMI Result",
                f"Monthly EMI = {emi:.2f}"
            )

        except:
            pass


# ==========================
# Main Program
# ==========================
if __name__ == "__main__":
    root = tk.Tk()
    app = AdvancedCalculator(root)
    root.mainloop()

In [38]:
import tkinter as tk
from tkinter import messagebox
import math

# ---------------- MAIN WINDOW ---------------- #
root = tk.Tk()
root.title("Ultimate Calculator")
root.geometry("480x600")
root.configure(bg="#1f1f1f")
root.resizable(False, False)

expression = ""
memory = 0

# ---------------- DISPLAY ---------------- #
display_var = tk.StringVar()

display = tk.Entry(
    root,
    textvariable=display_var,
    font=("Arial", 22),
    bd=8,
    relief="sunken",
    justify="right",
    bg="white"
)
display.pack(fill="x", padx=10, pady=10)

# ---------------- FUNCTIONS ---------------- #

def press(value):
    global expression
    expression += str(value)
    display_var.set(expression)


def clear():
    global expression
    expression = ""
    display_var.set("")


def backspace():
    global expression
    expression = expression[:-1]
    display_var.set(expression)


def equal():
    global expression
    try:
        expr = expression.replace("^", "**")
        result = eval(expr)
        display_var.set(result)
        expression = str(result)
    except:
        messagebox.showerror("Error", "Invalid Expression")
        clear()


def percent():
    global expression
    try:
        value = eval(expression)
        result = value / 100
        display_var.set(result)
        expression = str(result)
    except:
        messagebox.showerror("Error", "Invalid")


# ---------------- SCIENTIFIC ---------------- #

def scientific(func):
    global expression

    try:
        value = float(eval(expression))

        if func == "sin":
            ans = math.sin(math.radians(value))

        elif func == "cos":
            ans = math.cos(math.radians(value))

        elif func == "tan":
            ans = math.tan(math.radians(value))

        elif func == "sqrt":
            ans = math.sqrt(value)

        elif func == "log":
            ans = math.log10(value)

        elif func == "ln":
            ans = math.log(value)

        elif func == "square":
            ans = value ** 2

        elif func == "fact":
            ans = math.factorial(int(value))

        else:
            ans = value

        expression = str(ans)
        display_var.set(ans)

    except:
        messagebox.showerror("Error", "Invalid Input")


def insert_pi():
    press(str(math.pi))


def insert_e():
    press(str(math.e))


# ---------------- MEMORY ---------------- #

def memory_clear():
    global memory
    memory = 0


def memory_store():
    global memory
    try:
        memory = float(display_var.get())
    except:
        pass


def memory_recall():
    press(str(memory))


def memory_add():
    global memory
    try:
        memory += float(display_var.get())
    except:
        pass


def memory_sub():
    global memory
    try:
        memory -= float(display_var.get())
    except:
        pass


# ---------------- BUTTON STYLE ---------------- #

num_color = "#3498db"
op_color = "#f39c12"
sci_color = "#9b59b6"
mem_color = "#2ecc71"
clr_color = "#e74c3c"
eq_color = "#27ae60"

frame = tk.Frame(root, bg="#1f1f1f")
frame.pack()

buttons = [

["MC","MR","MS","M+","M-"],
["(",")","%","⌫","C"],
["7","8","9","/","sqrt"],
["4","5","6","*","square"],
["1","2","3","-","log"],
["0",".","=","+","ln"],
["sin","cos","tan","π","e"],
["x!","^","","",""]

]

for r,row in enumerate(buttons):
    for c,text in enumerate(row):

        if text=="":
            continue

        color=num_color

        if text in "+-*/=^":
            color=op_color

        if text in ["sin","cos","tan","sqrt","square","log","ln","π","e","x!"]:
            color=sci_color

        if text in ["MC","MR","MS","M+","M-"]:
            color=mem_color

        if text in ["C","⌫"]:
            color=clr_color

        if text=="=":
            color=eq_color

        btn=tk.Button(
            frame,
            text=text,
            width=8,
            height=2,
            font=("Arial",13,"bold"),
            bg=color,
            fg="white"
        )

        btn.grid(row=r,column=c,padx=3,pady=3)

        if text.isdigit() or text=="." or text in "()+-*/^":
            btn.config(command=lambda t=text:press(t))

        elif text=="=":
            btn.config(command=equal)

        elif text=="C":
            btn.config(command=clear)

        elif text=="⌫":
            btn.config(command=backspace)

        elif text=="%":
            btn.config(command=percent)

        elif text=="sqrt":
            btn.config(command=lambda:scientific("sqrt"))

        elif text=="square":
            btn.config(command=lambda:scientific("square"))

        elif text=="log":
            btn.config(command=lambda:scientific("log"))

        elif text=="ln":
            btn.config(command=lambda:scientific("ln"))

        elif text=="sin":
            btn.config(command=lambda:scientific("sin"))

        elif text=="cos":
            btn.config(command=lambda:scientific("cos"))

        elif text=="tan":
            btn.config(command=lambda:scientific("tan"))

        elif text=="π":
            btn.config(command=insert_pi)

        elif text=="e":
            btn.config(command=insert_e)

        elif text=="x!":
            btn.config(command=lambda:scientific("fact"))

        elif text=="MC":
            btn.config(command=memory_clear)

        elif text=="MR":
            btn.config(command=memory_recall)

        elif text=="MS":
            btn.config(command=memory_store)

        elif text=="M+":
            btn.config(command=memory_add)

        elif text=="M-":
            btn.config(command=memory_sub)

# Keyboard Support
root.bind("<Return>", lambda e: equal())
root.bind("<BackSpace>", lambda e: backspace())

root.mainloop()